# Module 5 - Evaluate, then optimize

You cannot improve what you do not measure. The golden set has **34 questions** (14 Syone, 20 Aurora) with expected keywords, expected documents and expected figures,
verified against the sources by `scripts/validate_dataset.py`.

Two evaluation modes:
* **retrieval** - no LLM: did the right documents and facts reach the generator? (instant, free, deterministic)
* **full** - generate the answer too; score facts, figure citation, refusals.

In [ ]:
import os, sys, json, asyncio, time, logging
from pathlib import Path
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)                                   # every path below is relative to the project root
os.environ.setdefault("HF_HOME", str(ROOT / "models" / "huggingface"))
logging.getLogger("neo4j").setLevel(logging.ERROR)
MODEL = os.getenv("FORMATION_MODEL", "formation-fast")   # use "formation-local" if you run LM Studio / Ollama

import pandas as pd
from backend.evaluation.golden_dataset import GOLDEN_EVALUATION_DATASET as GOLD
from backend.evaluation.langfuse_evals import LLMJudgeEvaluator
from backend.evaluation.opik_optimizer import OpikPromptOptimizer
from backend.graph_rag.pipeline import GraphRAGPipeline
from backend.graph_rag.neo4j_client import Neo4jGraphClient
from backend.graph_rag.qdrant_hybrid import QdrantHybridStore
from backend.okf_agents.deep_agent_harness import OKFDeepAgentHarness
from backend.okf_agents.pipeline import OKFAgentPipeline
neo4j, qdrant = Neo4jGraphClient(), QdrantHybridStore()
graph, okf, ev = GraphRAGPipeline(neo4j, qdrant), OKFAgentPipeline(), LLMJudgeEvaluator()
df = pd.DataFrame(GOLD)
display(df.groupby(["corpus", "category"]).size().unstack(0).fillna(0).astype(int))
display(df[["id", "category", "question"]].head(6))

## 1. Retrieval-only evaluation (no LLM, runs in seconds)

In [ ]:
def retrieval_eval(track):
    rows = []
    for item in GOLD:
        if track == "A: GraphRAG":
            ch = qdrant.hybrid_search(item["question"], corpus=item["corpus"], top_k=5)
            g = neo4j.query_graph_neighborhood(neo4j.extract_keywords(item["question"]), corpus=item["corpus"])
            docs, ctx = list(dict.fromkeys(c["doc_id"] for c in ch)), "\n".join(c["text"] for c in ch) + g["formatted_context"]
        else:
            nav = OKFDeepAgentHarness(item["corpus"]).scripted_navigation(item["question"])
            docs, ctx = [d["doc_id"] for d in nav["collected_documents"]], "\n".join(d["content"] for d in nav["collected_documents"])
        s = ev.evaluate_response(item, "", docs, ctx, "retrieval_only")
        rows.append({"id": item["id"], "category": item["category"], "track": track, "doc_found": s["retrieval_any_expected_doc"], "facts_in_context": s["context_keyword_recall"]})
    return pd.DataFrame(rows)

res = pd.concat([retrieval_eval("A: GraphRAG"), retrieval_eval("B: OKF scripted")])
res["ok"] = (res["doc_found"].fillna(True)) & (res["facts_in_context"].fillna(1) >= 0.6)
display(res.pivot_table(index="category", columns="track", values="ok", aggfunc="mean").round(2))
print("Overall:", res.groupby("track")["ok"].mean().round(3).to_dict())

Which questions does each track miss? (Read the failures: that is where the learning is.)

In [ ]:
display(res[~res["ok"]][["id", "track", "category", "doc_found", "facts_in_context"]].sort_values("id"))

## 2. Full evaluation (needs an LLM). Start small: 8 questions

In [ ]:
SUBSET = ["hw_001", "hw_002", "hw_003", "hw_007", "hw_010", "hw_020", "syone_002", "syone_014"]
async def full_eval(pipe, track):
    rows = []
    for item in [g for g in GOLD if g["id"] in SUBSET]:
        r = await pipe.execute_query(item["question"], corpus=item["corpus"], model_name=MODEL, include_context=True)
        s = ev.evaluate_response(item, r["answer"], r.get("retrieved_doc_ids"), r.get("context_text", ""), r["answer_mode"])
        rows.append({"id": item["id"], "track": track, "mode": r["answer_mode"], "passed": s["passed"], "keywords": s["found_keywords"],
                     "figure": s["visual_grounding_score"], "refusal_ok": s["refusal_ok"], "sec": r["latency_seconds"], "answer": r["answer"]})
    return pd.DataFrame(rows)

full = pd.concat([await full_eval(graph, "A: GraphRAG"), await full_eval(okf, "B: OKF agent")])
display(full.pivot(index="id", columns="track", values="passed"))
print(full.groupby("track")[["passed", "sec"]].agg({"passed": "mean", "sec": "mean"}).round(2))
if (full["mode"] == "extractive").any():
    print("\nSome answers were extractive: the LLM gateway was not reachable (see notebook 4). They count as failed.")

## 2b. LLM-as-a-judge: a panel of three
Keywords cannot tell a paraphrase from an error. A **judge** model reads the question, the reference answer and the candidate and returns a grade (0-1)
and a reason. One judge is just another model with its own biases, so three families grade the same Track A answers from section 2:
`formation-judge-deepseek` (the model that wrote the answers: watch for self-preference), `formation-judge-haiku` (Claude Haiku 5.5) and
`formation-judge-nemotron` (free, 50 requests/day). The judges are reasoning models: they think before they grade, so each grade takes a few seconds.

In [ ]:
from backend.evaluation.langfuse_evals import JUDGE_PANEL
rows = []
for _, r in full[(full["track"] == "A: GraphRAG") & (full["mode"] == "generated")].iterrows():
    item = next(g for g in GOLD if g["id"] == r["id"])
    panel = await ev.judge_panel(item, r["answer"])
    rows.append({"id": r["id"], "keywords_pass": r["passed"],
                 **{j.replace("formation-judge-", ""): (v["score"] if v else None) for j, v in panel["judges"].items()},
                 "spread": panel["judge_spread"]})
    for j, v in panel["judges"].items():
        print(f"{r['id']:10s} {j.replace('formation-judge-', ''):9s} {v['score'] if v else '-':>5}  {v['reason'][:110] if v else 'no verdict'}")
judged = pd.DataFrame(rows)
display(judged)

Read the rows where a judge and the keyword verdict disagree, and where the judges disagree with each other (`spread` 0.5 or more): sometimes the judge is right, sometimes the keyword list, and sometimes the judge is simply wrong. That is why you calibrate judges against a few answers graded by a human.

## 3. Optimize the prompt (a minimal version of what Opik's MetaPrompt / GEPA optimizers do)
Three prompt candidates are scored on the **real pipeline**; the best one wins. Opik's optimizers add an LLM that proposes and mutates candidates for you.

In [ ]:
opt = OpikPromptOptimizer(ev)
try:
    out = await opt.run_optimization_loop(graph, MODEL, ids=["hw_001", "hw_003", "hw_007", "hw_012", "syone_002", "syone_007"])
    for h in out["history"]: print(f"{h['variant']:28s} passed {h['passed']}")
    print("\nbest:", out["best_variant"])
except Exception as e:
    print("Needs a working LLM:", e)

### Going further with Opik itself (optional)
```python
pip install opik-optimizer           # then, with an Opik workspace key in .env (OPIK_API_KEY):
from opik_optimizer import MetaPromptOptimizer, ChatPrompt
```
Wrap `GraphRAGPipeline.execute_query` as the task function and `LLMJudgeEvaluator.evaluate_response` as the metric; see Opik's docs for the exact signatures of the release you installed.

### Exercises
1. Which category does Track A fail most in the retrieval table? Which does Track B? Why?
1. In section 2b, find an answer where the judges disagree. Read it yourself: who is right?
2. Add two golden questions of your own to `backend/evaluation/golden_dataset.py`; run `python scripts/validate_dataset.py` to confirm they are answerable.
3. Add a fourth prompt variant that asks for a one-sentence answer. Does the pass rate change?